# EDA bảng RFM khách hàng United Kingdom

Notebook này khám phá bảng `data/processed/rfm_uk_full.csv` đã được tạo từ toàn bộ dữ liệu giao dịch của United Kingdom.

Mục tiêu của bước này là hiểu phân phối của ba chỉ số `Recency`, `Frequency`, `Monetary`, phát hiện các khách hàng có hành vi khác biệt và kiểm tra tác động của log-transform trước khi chuyển sang K-Means.

## 1. Load bảng RFM

Bảng RFM đang ở cấp độ khách hàng: mỗi dòng là một `CustomerID` cùng ba chỉ số hành vi mua hàng.

Đoạn code dưới đây import các thư viện cần thiết, đọc `data/processed/rfm_uk_full.csv` và hiển thị vài dòng đầu tiên để kiểm tra cấu trúc dữ liệu.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RFM_PATH = PROJECT_ROOT / "data" / "processed" / "rfm_uk_full.csv"

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.labelsize"] = 12

rfm = pd.read_csv(RFM_PATH)

rfm.head()

## 2. Thống kê mô tả RFM

Trước khi vẽ biểu đồ, cần xem các giá trị trung bình, trung vị, min, max và độ lệch của từng biến. Đây là bước giúp nhận diện nhanh biến nào có đuôi dài hoặc khách hàng cực trị.

Đoạn code dưới đây tạo bảng tổng quan quy mô RFM và kiểm tra dữ liệu thiếu ở ba chỉ số chính.

In [ ]:
rfm_overview = pd.DataFrame(
    {
        "Metric": ["Rows", "Customers", "Missing Values"],
        "Value": [
            f"{len(rfm):,}",
            f"{rfm['CustomerID'].nunique():,}",
            f"{int(rfm.isna().sum().sum()):,}",
        ],
    }
)

rfm_overview

Đoạn code dưới đây tính thống kê mô tả cho `Recency`, `Frequency` và `Monetary`, gồm trung bình, trung vị, phân vị và giá trị lớn nhất.

In [ ]:
rfm_describe = rfm[["Recency", "Frequency", "Monetary"]].describe().T
rfm_describe["median"] = rfm[["Recency", "Frequency", "Monetary"]].median()
rfm_describe["skewness"] = rfm[["Recency", "Frequency", "Monetary"]].skew()

rfm_describe.round(2)

**Nhận xét:** bảng RFM có 3,916 khách hàng UK. `Recency` có trung vị 51 ngày và lớn nhất 374 ngày. `Frequency` có trung vị 2 hóa đơn nhưng cao nhất 206 hóa đơn. `Monetary` có trung vị 644.10 GBP nhưng cao nhất 259,657.30 GBP, cho thấy dữ liệu có độ lệch mạnh ở nhóm khách hàng giá trị cao.

## 3. Phân phối ba chỉ số RFM

Histogram giúp xem phần lớn khách hàng đang tập trung ở vùng giá trị nào và biến nào có đuôi kéo dài.

Đoạn code dưới đây vẽ histogram cho `Recency`, `Frequency` và `Monetary` ở thang đo gốc.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ["Recency", "Frequency", "Monetary"]):
    sns.histplot(rfm[col], bins=40, ax=ax, color="#4c72b0")
    ax.axvline(rfm[col].mean(), color="#c44e52", linestyle="--", label="Mean")
    ax.axvline(rfm[col].median(), color="#55a868", linestyle="-", label="Median")
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Customers")
    ax.legend()

plt.tight_layout()
plt.show()

**Nhận xét:** `Frequency` và `Monetary` lệch phải rất mạnh: 75% khách hàng có tối đa 5 hóa đơn và chi tiêu không quá 1,567.53 GBP, nhưng vẫn có khách đạt 206 hóa đơn và 259,657.30 GBP. `Recency` cũng có đuôi dài hơn bản RFM cũ khi giá trị lớn nhất lên tới 374 ngày.

## 4. Boxplot và outlier theo IQR

Boxplot giúp nhìn rõ các điểm nằm ngoài vùng phổ biến. Trong bài toán khách hàng, outlier không nhất thiết là dữ liệu sai; nhiều khi đó chính là khách hàng VIP, khách bán buôn hoặc khách đã lâu không quay lại.

Đoạn code dưới đây vẽ boxplot cho ba biến RFM để quan sát độ phân tán và các điểm cực trị.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ["Recency", "Frequency", "Monetary"]):
    sns.boxplot(y=rfm[col], ax=ax, color="#dd8452")
    ax.set_title(f"Boxplot of {col}")
    ax.set_ylabel(col)

plt.tight_layout()
plt.show()

Đoạn code dưới đây tính ngưỡng outlier theo quy tắc IQR cho từng biến. Mục tiêu là định lượng có bao nhiêu khách hàng nằm ngoài vùng phổ biến.

In [ ]:
outlier_rows = []

for col in ["Recency", "Frequency", "Monetary"]:
    q1 = rfm[col].quantile(0.25)
    q3 = rfm[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = int(((rfm[col] < lower) | (rfm[col] > upper)).sum())

    outlier_rows.append(
        {
            "Feature": col,
            "Q1": q1,
            "Median": rfm[col].median(),
            "Q3": q3,
            "IQR": iqr,
            "LowerBound": lower,
            "UpperBound": upper,
            "OutlierCustomers": count,
            "OutlierPct": count / len(rfm) * 100,
        }
    )

outlier_summary = pd.DataFrame(outlier_rows)
outlier_summary.round(2)

**Nhận xét:** theo IQR, `Recency` có 122 khách hàng outlier, tương đương 3.12%. `Frequency` có 256 khách hàng outlier, tương đương 6.54%. `Monetary` có 379 khách hàng outlier, tương đương 9.68%. Các outlier này không nên xóa tự động vì có thể đại diện cho nhóm khách hàng có giá trị hoặc hành vi đặc biệt.

## 5. Khách hàng đặc biệt

Phần này xem trực tiếp các khách hàng có giá trị cực trị theo từng chiều RFM. Đây là cách kiểm tra xem outlier có thể mang ý nghĩa kinh doanh hay không.

Đoạn code dưới đây lấy top khách hàng có `Frequency` cao nhất, tức nhóm quay lại mua nhiều lần nhất.

In [ ]:
top_frequency_customers = rfm.sort_values("Frequency", ascending=False).head(10)
top_frequency_customers

**Nhận xét:** khách hàng `12748` có Frequency cao nhất với 206 hóa đơn, tiếp theo là khách `17841` với 124 hóa đơn. Đây là các khách hàng có mức độ quay lại rất cao so với trung vị toàn bộ tập chỉ là 2 hóa đơn.

Đoạn code dưới đây lấy top khách hàng có `Monetary` cao nhất, tức nhóm đóng góp doanh thu lớn nhất.

In [ ]:
top_monetary_customers = rfm.sort_values("Monetary", ascending=False).head(10)
top_monetary_customers

**Nhận xét:** khách hàng `18102` có Monetary cao nhất với 259,657.30 GBP, tiếp theo là `17450` với 194,390.79 GBP. Đáng chú ý, khách `16446` chỉ có 2 hóa đơn nhưng chi tiêu tới 168,472.50 GBP, cho thấy giá trị khách hàng không chỉ đến từ số lần mua.

Đoạn code dưới đây lấy top khách hàng có `Recency` cao nhất, tức những khách đã lâu nhất chưa quay lại mua hàng.

In [ ]:
top_recency_customers = rfm.sort_values("Recency", ascending=False).head(10)
top_recency_customers

**Nhận xét:** có nhiều khách hàng đạt Recency 374 ngày, nghĩa là gần như không quay lại trong suốt phần lớn giai đoạn quan sát. Đây là nhóm quan trọng cần được nhận diện khi phân cụm vì có thể thuộc nhóm inactive hoặc at-risk.

## 6. Quan hệ giữa các chỉ số RFM

Các scatter plot giúp xem mối quan hệ giữa tần suất mua, tổng chi tiêu và thời gian mua gần nhất. Đây là phần chuẩn bị trực quan trước khi đưa dữ liệu vào K-Means.

Đoạn code dưới đây tính ma trận tương quan giữa ba chỉ số RFM.

In [ ]:
rfm_correlation = rfm[["Recency", "Frequency", "Monetary"]].corr()
rfm_correlation.round(3)

Đoạn code dưới đây vẽ scatter plot giữa `Frequency` và `Monetary`. Biểu đồ này cho thấy khách mua nhiều lần có thường đóng góp nhiều doanh thu hơn hay không.

In [ ]:
ax = sns.scatterplot(data=rfm, x="Frequency", y="Monetary", alpha=0.65)
ax.set_title("Frequency vs Monetary")
ax.set_xlabel("Frequency")
ax.set_ylabel("Monetary (GBP)")
plt.tight_layout()
plt.show()

**Nhận xét:** `Frequency` và `Monetary` có tương quan dương khoảng 0.508. Điều này hợp lý vì khách mua nhiều lần thường có tổng chi tiêu cao hơn, nhưng vẫn có ngoại lệ như khách `16446` có Frequency thấp nhưng Monetary rất cao.

Đoạn code dưới đây vẽ scatter plot giữa `Recency` và `Monetary`. Biểu đồ này giúp phân biệt khách vừa mua gần đây với khách đã lâu không quay lại nhưng từng chi tiêu cao.

In [ ]:
ax = sns.scatterplot(data=rfm, x="Recency", y="Monetary", alpha=0.65, color="#55a868")
ax.set_title("Recency vs Monetary")
ax.set_xlabel("Recency")
ax.set_ylabel("Monetary (GBP)")
plt.tight_layout()
plt.show()

**Nhận xét:** `Recency` và `Monetary` có tương quan âm nhẹ khoảng -0.129. Nói cách khác, khách mua gần đây có xu hướng chi tiêu cao hơn một chút, nhưng quan hệ này không mạnh bằng quan hệ giữa Frequency và Monetary.

Đoạn code dưới đây vẽ scatter plot giữa `Recency` và `Frequency`. Biểu đồ này giúp xem khách mua thường xuyên có thường quay lại gần đây hơn không.

In [ ]:
ax = sns.scatterplot(data=rfm, x="Recency", y="Frequency", alpha=0.65, color="#c44e52")
ax.set_title("Recency vs Frequency")
ax.set_xlabel("Recency")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()

**Nhận xét:** `Recency` và `Frequency` có tương quan âm khoảng -0.275. Điều này cho thấy khách mua nhiều lần thường có xu hướng quay lại gần đây hơn, nhưng dữ liệu vẫn có nhiều nhóm hành vi khác nhau nên cần phân cụm thay vì chỉ nhìn từng cặp biến.

## 7. So sánh trước và sau log-transform

K-Means dùng khoảng cách Euclid, nên các biến lệch phải mạnh có thể kéo tâm cụm về phía các khách hàng cực trị. Log-transform giúp giảm độ lệch mà không xóa bỏ khách hàng giá trị cao.

Đoạn code dưới đây tạo phiên bản log-transform bằng `np.log1p` cho cả ba biến RFM.

In [ ]:
rfm_log = rfm[["Recency", "Frequency", "Monetary"]].apply(np.log1p)
rfm_log.columns = ["LogRecency", "LogFrequency", "LogMonetary"]

rfm_log.head()

Đoạn code dưới đây so sánh độ lệch trước và sau log-transform để xem phép biến đổi có làm phân phối cân bằng hơn không.

In [ ]:
skew_comparison = pd.DataFrame(
    {
        "Feature": ["Recency", "Frequency", "Monetary"],
        "SkewBeforeLog": rfm[["Recency", "Frequency", "Monetary"]].skew().values,
        "SkewAfterLog": rfm_log.skew().values,
    }
)

skew_comparison.round(2)

**Nhận xét:** log-transform làm độ lệch giảm rõ rệt. `Frequency` giảm từ 10.64 xuống 1.18, `Monetary` giảm từ 20.45 xuống 0.37, và `Recency` chuyển từ 1.24 xuống -0.32. Vì vậy, với RFM full-period, việc log cả ba biến là hợp lý trước khi chuẩn hóa và chạy K-Means.

Đoạn code dưới đây vẽ histogram của ba biến sau log-transform để quan sát phân phối mới.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, rfm_log.columns):
    sns.histplot(rfm_log[col], bins=40, ax=ax, color="#8172b3")
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Customers")

plt.tight_layout()
plt.show()

## 8. Kết luận EDA RFM

Bảng RFM full-period của UK có 3,916 khách hàng, nhiều hơn file RFM cũ vì đã dùng toàn bộ giai đoạn dữ liệu. Các chỉ số RFM có độ lệch rõ rệt, đặc biệt là `Frequency` và `Monetary`.

`Frequency` có trung vị 2 hóa đơn nhưng cao nhất 206 hóa đơn. `Monetary` có trung vị 644.10 GBP nhưng cao nhất 259,657.30 GBP. Điều này cho thấy phần lớn khách hàng nằm ở vùng mua ít hoặc chi tiêu vừa phải, trong khi một nhóm nhỏ có giá trị vượt trội.

`Recency` có giá trị lớn nhất 374 ngày, phản ánh sự xuất hiện của nhóm khách lâu chưa quay lại. Đây là khác biệt quan trọng so với RFM cũ và cần được giữ lại để K-Means nhận diện nhóm inactive hoặc at-risk.

Từ các phân tích trên, phase modeling nên dùng pipeline: `log1p(Recency)`, `log1p(Frequency)`, `log1p(Monetary)` rồi chuẩn hóa bằng `StandardScaler` trước khi chạy K-Means.